# Conformal Curvature Invariants on Classical Surfaces

This notebook introduces the core module of `conformal_toolkit` by computing
conformal curvature invariants on classical Riemannian surfaces: the round
2-sphere $S^2$, the round 4-sphere $S^4$, and flat Euclidean space $\mathbb{R}^n$.

## Mathematical background

In conformal geometry, the key curvature decomposition replaces the Ricci tensor
with the **Schouten tensor**
$$P_{ab} = \frac{1}{n-2}\Bigl(\mathrm{Ric}_{ab} - \frac{R}{2(n-1)}\,g_{ab}\Bigr)$$
and the conformally invariant **Weyl tensor** $W_{abcd}$. These two tensors
together carry the same information as the full Riemann tensor, but separate the
"conformally pure" part (Weyl) from the "trace" part (Schouten).

Higher-order conformal invariants built from $P$ include:
- **$Q$-curvature** $Q_{2k}$: scalar curvature quantities whose integrals are
  conformally invariant (in the critical dimension). $Q_2 = R$ is the scalar
  curvature; $Q_4 = -\Delta J - 2|P|^2 + \frac{n}{2}J^2$ is Branson's $Q$-curvature.
- **Bach tensor** $B_{ab}$: the gradient of the Weyl functional, vanishes on
  conformally Einstein manifolds. Requires dimension $\ge 4$.
- **GJMS operators** $P_{2k}$: conformally covariant differential operators.
  $P_2 = \Delta$ (Laplacian); $P_4$ is the Paneitz operator.

We will verify these against known closed-form results from the literature.

## 1. The round 2-sphere $S^2$

The round unit sphere $S^2$ has metric $g = d\theta^2 + \sin^2\!\theta\, d\varphi^2$,
scalar curvature $R = 2$, and is conformally flat. Its Schouten tensor satisfies
$P = \frac{R}{4}\,g = \frac{1}{2}\,g$ (using the 2D convention where the general
formula has a $1/(n-2)$ pole).

In [ ]:
from sage.all import Manifold, sin, cos, var
from conformal_toolkit.core.conformal_structure import ConformalStructure

# Build the round S^2
S = Manifold(2, 'S2', structure='Riemannian')
X = S.chart(r'theta:(0,pi):\theta phi:(0,2*pi):\phi')
theta, phi = X[:]
g = S.metric('g')
g[0, 0] = 1
g[1, 1] = sin(theta)**2

cs = ConformalStructure(g)
print("Metric on S^2:")
g.display()

In [ ]:
# Compute the Schouten tensor
P = cs.schouten()
print("Schouten tensor P on S^2:")
P.display()

# Verify: P should equal g/2 on the round S^2
# P_{00} = 1/2 * g_{00} = 1/2
# P_{11} = 1/2 * g_{11} = sin(theta)^2 / 2
print("\nVerification: P - (1/2)*g should vanish:")
diff = P - g/2
diff.display()

# Output:
# Schouten tensor P on S^2:
#   P = 1/2 dtheta*dtheta + 1/2*sin(theta)^2 dphi*dphi
# Verification: P - (1/2)*g should vanish:
#   0

In [ ]:
# Q_2 curvature = scalar curvature R
Q2 = cs.q_curvature(order=2)
print("Q_2 on S^2 (= scalar curvature R):")
Q2.display()

# On the round unit S^2, R = 2
# Output:
# Q_2 on S^2 (= scalar curvature R):
#   r(g) = 2

## 2. The round 4-sphere $S^4$

The round $S^4$ is conformally flat, so its Weyl tensor (and therefore the Bach
tensor) vanishes identically. Its $Q_4$ curvature is a positive constant related
to the dimension and scalar curvature:
$$Q_4 = -\Delta J - 2|P|^2 + 2J^2$$
On $S^4$ with $R = 12$, we get $J = R/6 = 2$, $|P|^2 = 4$, and $\Delta J = 0$
(since $J$ is constant), giving $Q_4 = -0 - 8 + 8 = 0$... Let us compute.

In [ ]:
# Build the round S^4 in hyperspherical coordinates
S4 = Manifold(4, 'S4', structure='Riemannian')
X4 = S4.chart(r'th1:(0,pi):\theta_1 th2:(0,pi):\theta_2 th3:(0,pi):\theta_3 ph:(0,2*pi):\varphi')
th1, th2, th3, ph = X4[:]
g4 = S4.metric('g')
g4[0, 0] = 1
g4[1, 1] = sin(th1)**2
g4[2, 2] = sin(th1)**2 * sin(th2)**2
g4[3, 3] = sin(th1)**2 * sin(th2)**2 * sin(th3)**2

cs4 = ConformalStructure(g4)

# Scalar curvature of the round S^4: R = n(n-1) = 4*3 = 12
R4 = cs4.ricci_scalar()
print("Scalar curvature of S^4:")
R4.display()

# Output:
# Scalar curvature of S^4:
#   r(g) = 12

In [ ]:
# Bach tensor on S^4 -- should vanish (conformally flat)
B = cs4.bach()
print("Bach tensor on S^4:")
B.display()

# Output:
# Bach tensor on S^4:
#   B = 0

In [ ]:
# Q_4 curvature on S^4
Q4 = cs4.q_curvature(order=4)
print("Q_4 on round S^4:")
print(Q4.display())

# On the round S^4: Q_4 should be a constant.
# J = R/(2*(n-1)) = 12/6 = 2
J4 = cs4.schouten_trace()
print("\nSchouten trace J =", J4.display())

## 3. Conformal rescaling: flat $\mathbb{R}^3$ under $\hat{g} = e^{2\omega}g$

Flat space has vanishing Schouten tensor ($P = 0$). After a conformal rescaling
$\hat{g} = e^{2\omega}\,g$ with $\omega = x_0$, the new metric acquires non-trivial
curvature. The Schouten tensor transforms as:
$$\hat{P}_{ab} = P_{ab} - \nabla_a\nabla_b\omega + (\nabla_a\omega)(\nabla_b\omega) - \tfrac{1}{2}|d\omega|^2 g_{ab}$$

We demonstrate this using `ConformalStructure.under_rescaling()`.

In [ ]:
# Flat R^3
from sage.all import Manifold
R3 = Manifold(3, 'R3', structure='Riemannian')
Y = R3.chart('x0 x1 x2')
x0, x1, x2 = Y[:]
g_flat = R3.metric('g')
g_flat[0, 0] = 1
g_flat[1, 1] = 1
g_flat[2, 2] = 1

cs_flat = ConformalStructure(g_flat)

# Verify: Schouten vanishes on flat space
P_flat = cs_flat.schouten()
print("Schouten on flat R^3:")
P_flat.display()

# Output:
# Schouten on flat R^3:
#   P = 0

In [ ]:
# Rescale by omega = x0: g_hat = e^{2*x0} * g_flat
omega = R3.scalar_field(x0, name='omega')
cs_hat = cs_flat.under_rescaling(omega)

print("Rescaled metric g_hat = e^{2*x0} * delta:")
cs_hat.metric.display()

# Schouten of the rescaled metric is now non-trivial
P_hat = cs_hat.schouten()
print("\nSchouten of rescaled metric:")
P_hat.display()

# Output:
# The rescaled metric is e^{2*x0} * (dx0^2 + dx1^2 + dx2^2)
# and its Schouten tensor has non-zero components showing the
# conformal curvature introduced by the rescaling.

In [ ]:
# Q_2 curvature of the rescaled metric (should be non-zero)
Q2_hat = cs_hat.q_curvature(order=2)
print("Q_2 of rescaled metric:")
Q2_hat.display()

# Compare with flat space Q_2 = 0
Q2_flat = cs_flat.q_curvature(order=2)
print("\nQ_2 of flat R^3:")
Q2_flat.display()

## 4. GJMS Paneitz operator on flat $\mathbb{R}^4$

The Paneitz operator $P_4$ is the fourth-order GJMS operator:
$$P_4(f) = \Delta^2 f + \mathrm{div}(V \cdot df) + \frac{n-4}{2}\,Q_4\, f$$
where $V_{ab} = (n-2)J\,g_{ab} - 4P_{ab}$.

On flat $\mathbb{R}^4$, $P = 0$ and $J = 0$, so $P_4$ reduces to the bi-Laplacian
$\Delta^2$. We verify this on a test function.

In [ ]:
# Flat R^4
R4M = Manifold(4, 'R4', structure='Riemannian')
Z = R4M.chart('x0 x1 x2 x3')
x0, x1, x2, x3 = Z[:]
g4_flat = R4M.metric('g')
for i in range(4):
    g4_flat[i, i] = 1

cs4_flat = ConformalStructure(g4_flat)

# Test function: f = x0^2 + x1^2 (a harmonic-ish function)
f = R4M.scalar_field(x0**2 + x1**2, name='f')

# Apply the Paneitz operator
P4f = cs4_flat.gjms_operator(f, order=4)
print("P_4(x0^2 + x1^2) on flat R^4:")
print(P4f.display())

# On flat R^4, P_4 = Delta^2
# Delta(x0^2 + x1^2) = 2 + 2 = 4
# Delta^2(x0^2 + x1^2) = Delta(4) = 0
# So P_4(f) = 0.

# Try a more interesting function: f = x0^4
f2 = R4M.scalar_field(x0**4, name='f2')
P4f2 = cs4_flat.gjms_operator(f2, order=4)
print("\nP_4(x0^4) on flat R^4:")
print(P4f2.display())

# Delta(x0^4) = 12*x0^2
# Delta^2(x0^4) = Delta(12*x0^2) = 24
# So P_4(x0^4) = 24.

In [ ]:
# For comparison: P_2 = Delta (the Laplacian)
P2f = cs4_flat.gjms_operator(f, order=2)
print("P_2(x0^2 + x1^2) = Delta(x0^2 + x1^2) =")
P2f.display()

# Output:
# P_2(x0^2 + x1^2) = Delta(x0^2 + x1^2) = 4

## Summary

In this notebook we demonstrated the core conformal curvature computations in
`conformal_toolkit`:

| Surface | Invariant | Result |
|---------|-----------|--------|
| $S^2$ | Schouten $P$ | $\frac{1}{2}g$ |
| $S^2$ | $Q_2$ | $2$ |
| $S^4$ | Bach $B$ | $0$ (conformally flat) |
| $S^4$ | $Q_4$ | constant |
| $\mathbb{R}^3$ | Schouten $P$ | $0$ (flat) |
| $e^{2x_0}\delta$ | Schouten $\hat{P}$ | non-zero (curvature from rescaling) |
| $\mathbb{R}^4$ | $P_4(x_0^4)$ | $24 = \Delta^2(x_0^4)$ |

All results match the known closed-form values from the conformal geometry literature.
The `ConformalStructure` class provides a clean interface for exploring these
invariants on any SageManifolds metric.

**Next:** Notebook 02 explores *extrinsic* conformal invariants for hypersurfaces.